In [25]:
import joblib
import pickle
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [26]:
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [36]:
tfidf_fakenews = joblib.load('tfidf_fakenews.pkl')
tfidf_categories = joblib.load('tfidf_categories.pkl')
le = joblib.load('le_categories.pkl')
model_fakenews = pickle.load(open('model_fakenews.pkl','rb'))
model_categories = pickle.load(open('model_categories2.pkl','rb'))
model_fakenews.multi_class = 'auto'
model_categories.multi_class = 'auto'

In [37]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

## Функция предсказания

In [47]:
def predict(title, text=''):
    full_text = (title + ' ' + text).strip()
    processed = preprocess(full_text)


    x_fake = tfidf_fakenews.transform([processed])
    fake_label = model_fakenews.predict(x_fake)[0]
    decision = model_fakenews.decision_function(x_fake)[0]
    fk = 1 / (1 + np.exp(-decision))


    x_cat = tfidf_categories.transform([processed])
    cat_label = model_categories.predict(x_cat)[0]
    cat_name = le.inverse_transform([cat_label])[0]

    if fake_label == 1:
        fake_verdict = f"Да, это фейк (наша модерация уверена на {fk})"
    else:
        fake_verdict = f" Нет, новость подтверждена (вероятность фейка равна {fk})"

    print(f"Статья: {full_text}")
    print(f"Фейк: {fake_verdict}")
    print(f"Категория: {cat_name}")
    return fake_label, cat_name



Тест:

In [50]:
print("АМЕРИКАНСКИЙ МАКС")
print("Проверим вашу новость!")
print("-" * 60)
title = input("Заголовок: ")
text  = input("Текст: ")
print("\nИтог")
print("-" * 40)
predict(title=title, text=text)

АМЕРИКАНСКИЙ МАКС
Проверим вашу новость!
------------------------------------------------------------
Заголовок: олтжор
Текст: иоритзо

Итог
----------------------------------------
Статья: олтжор иоритзо
Фейк: Да, это фейк (наша модерация уверена на 0.942016527413915)
Категория: ENTERTAINMENT


(np.float64(1.0), 'ENTERTAINMENT')